<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0f0f 0%, #1a1a1a 100%); border: 2px solid #a855f7; border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(168, 85, 247, 0.2); font-family: sans-serif;'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ Clone, Transcreva e Gere Áudios Ilimitados</h1>
  <h3 style='color: #d8b4fe; margin: 0 0 15px 0; font-weight: 400;'><strong>Mod Por: Railson devs Criador: Lucro A Lucro</strong></h3>
  <div style='display: flex; justify-content: center; gap: 15px;'>
    <a href="http://youtube.com/@canalwfreitas" target="_blank" style="background: #FF0000; color: white; padding: 8px 16px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 14px;">▶ YouTube</a>
    <a href="https://www.instagram.com/0xwsf/" target="_blank" style="background: linear-gradient(45deg, #f09433 0%, #e6683c 25%, #dc2743 50%, #cc2366 75%, #bc1888 100%); color: white; padding: 8px 16px; border-radius: 8px; text-decoration: none; font-weight: bold; font-size: 14px;">📸 Instagram</a>
  </div>
</div>

> ⚠️ Certifique-se de selecionar **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU** antes de rodar.


In [ ]:
#@title 🚀 INICIAR GERADOR PREMIUM COM CLOUDFLARE, STATUS, HISTÓRICO E GEMINI (44.1kHz)
# ============================================================
# CONFIGURAÇÃO DO SEU TEMPLATE
# ============================================================
# 1) Suba set_api.php e api_url.json no seu site.
# 2) Coloque aqui a URL do set_api.php.
# 3) Use o mesmo token configurado no set_api.php.
# 4) Opcional: coloque sua chave Gemini para o botão "Melhorar Texto com IA".

SET_API_URL = "https://SEU-SITE.com/set_api.php"  # exemplo: https://meusite.com/set_api.php
SET_API_TOKEN = "SENHA_FORTE_AQUI"

# Gemini é opcional. Se deixar vazio, a API de voz continua funcionando.
GEMINI_API_KEY = ""  # cole sua chave do Google AI Studio aqui, se quiser usar o botão IA
GEMINI_MODEL = "gemini-2.5-flash"

# Monitor mais tolerante para não recriar túnel durante geração de voz.
HEALTH_INTERVAL = 90       # segundos entre verificações
HEALTH_TIMEOUT = 30        # timeout do teste /health
MAX_FAILS = 5              # só recria após várias falhas seguidas
RECREATE_DELAY = 35        # espera antes de recriar túnel
CLOUDFLARE_CREATE_TIMEOUT = 90

# Histórico e limpeza automática
OUTPUT_DIR = "/tmp/omnivoice_outputs"
VOICES_DIR = "/tmp/omnivoice_voices"
WHISPER_MODEL_NAME = "small"  # tiny/base/small/medium; small é bom equilíbrio no Colab
OUTPUT_TTL_SECONDS = 60 * 60       # apaga WAVs com mais de 1 hora
MAX_HISTORY_ITEMS = 30             # mantém somente os últimos 30 registros em memória

# ============================================================
# NÃO ALTERE ABAIXO, A MENOS QUE SAIBA O QUE ESTÁ FAZENDO
# ============================================================

import IPython.display as display
from IPython.display import clear_output
from IPython.utils import io
import os, sys, time, threading, subprocess, re, queue, signal, json, uuid, asyncio, glob, shutil
from datetime import datetime
from typing import Optional

display.display(display.HTML("""
<div style='background:#080808;padding:48px;border-radius:16px;text-align:center;
            font-family:system-ui,sans-serif;max-width:640px;margin:0 auto;border:1px solid #7c3aed;
            box-shadow:0 18px 50px rgba(124,58,237,.22);'>
  <div style='width:32px;height:32px;border:2px solid #2a154f;border-top-color:#c4b5fd;
              border-radius:50%;animation:spin .8s linear infinite;margin:0 auto 20px;'></div>
  <style>@keyframes spin{to{transform:rotate(360deg)}}</style>
  <p style='color:#d8b4fe;font-size:14px;letter-spacing:.5px;margin:0;'>
    Instalando dependências, carregando OmniVoice, ativando cache e preparando Cloudflare Tunnel...
  </p>
</div>
"""))

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

with io.capture_output() as captured:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers>=5.3"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice", "pydub", "fastapi", "uvicorn[standard]", "python-multipart", "scipy", "requests", "openai-whisper"])

    # Instala cloudflared direto do release oficial.
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run([
            "wget", "-q", "-O", "/tmp/cloudflared",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        ], check=True)
        subprocess.run(["chmod", "+x", "/tmp/cloudflared"], check=True)
        subprocess.run(["mv", "/tmp/cloudflared", "/usr/local/bin/cloudflared"], check=True)

    import torch, numpy as np, io as _io, wave, base64, requests, whisper
    from omnivoice import OmniVoice, OmniVoiceGenerationConfig
    from fastapi import FastAPI, File, UploadFile, Form, Body
    from fastapi.responses import JSONResponse, FileResponse
    from fastapi.middleware.cors import CORSMiddleware
    import uvicorn
    from scipy.signal import resample

    # CACHE DO MODELO:
    # O OmniVoice é carregado apenas uma vez nesta célula e fica em memória/GPU.
    # Cada requisição /api/gerar reutiliza este objeto global.
    model = OmniVoice.from_pretrained(
        'k2-fsa/OmniVoice',
        device_map='cuda', dtype=torch.float16,
        load_asr=True, token=False,
    )
    sampling_rate = model.sampling_rate

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VOICES_DIR, exist_ok=True)

# Controle para o monitor não reiniciar túnel enquanto OmniVoice está gerando.
IS_GENERATING = False
LAST_API_ENDPOINT_SENT = None
MODEL_CACHE_READY = True
GENERATION_LOCK = threading.Lock()
TRANSCRIPTION_LOCK = threading.Lock()
WHISPER_MODEL = None

CURRENT_JOB = {
    "id": None,
    "active": False,
    "status": "idle",
    "message": "Aguardando geração.",
    "progress": 0,
    "started_at": None,
    "finished_at": None,
    "mode": None,
}

HISTORY = []

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def set_job(status="idle", message="", progress=0, job_id=None, mode=None):
    """Atualiza status global para o template consultar em /api/status."""
    CURRENT_JOB.update({
        "id": job_id if job_id is not None else CURRENT_JOB.get("id"),
        "active": status not in ("idle", "done", "error"),
        "status": status,
        "message": message,
        "progress": int(progress),
        "mode": mode if mode is not None else CURRENT_JOB.get("mode"),
    })
    if status in ("preparing", "generating"):
        CURRENT_JOB["started_at"] = CURRENT_JOB.get("started_at") or now_str()
        CURRENT_JOB["finished_at"] = None
    if status in ("done", "error", "idle"):
        CURRENT_JOB["finished_at"] = now_str()

def cleanup_outputs():
    """Limpa WAVs antigos e reduz histórico para evitar acumular lixo no Colab."""
    now = time.time()
    for path in glob.glob(os.path.join(OUTPUT_DIR, "*.wav")):
        try:
            if now - os.path.getmtime(path) > OUTPUT_TTL_SECONDS:
                os.remove(path)
        except Exception:
            pass

    while len(HISTORY) > MAX_HISTORY_ITEMS:
        old = HISTORY.pop()
        try:
            p = old.get("file_path")
            if p and os.path.exists(p):
                os.remove(p)
        except Exception:
            pass

def get_whisper_model():
    """Carrega Whisper sob demanda e mantém em cache para transcrição automática."""
    global WHISPER_MODEL
    if WHISPER_MODEL is None:
        set_job("transcribing", f"Carregando Whisper {WHISPER_MODEL_NAME} para transcrição...", 10)
        WHISPER_MODEL = whisper.load_model(WHISPER_MODEL_NAME)
    return WHISPER_MODEL

def normalize_language_code(language):
    if not language or language == "Auto":
        return None
    return language.split("(")[-1].rstrip(")").strip() if "(" in language else language

def transcribe_audio_file(audio_path, language="Auto"):
    """Transcreve áudio para texto usando Whisper. Retorna string limpa."""
    with TRANSCRIPTION_LOCK:
        set_job("transcribing", "Transcrevendo áudio com Whisper...", 25)
        wm = get_whisper_model()
        lang_code = normalize_language_code(language)
        kwargs = {"fp16": torch.cuda.is_available()}
        if lang_code:
            kwargs["language"] = lang_code
        result = wm.transcribe(audio_path, **kwargs)
        text = (result.get("text") or "").strip()
        set_job("transcribing", "Transcrição concluída.", 100)
        return text

def voice_info_path(voice_id):
    return os.path.join(VOICES_DIR, voice_id, "info.json")

def voice_ref_path(voice_id):
    folder = os.path.join(VOICES_DIR, voice_id)
    matches = glob.glob(os.path.join(folder, "ref.*"))
    return matches[0] if matches else None

def list_saved_voices():
    items = []
    for folder in sorted(glob.glob(os.path.join(VOICES_DIR, "*"))):
        info_file = os.path.join(folder, "info.json")
        if not os.path.exists(info_file):
            continue
        try:
            with open(info_file, "r", encoding="utf-8") as f:
                info = json.load(f)
            ref = voice_ref_path(info.get("id", ""))
            info["has_ref_audio"] = bool(ref and os.path.exists(ref))
            items.append(info)
        except Exception:
            pass
    return items

def save_voice_profile(name, ref_audio_path, ref_text="", language="Auto"):
    """Salva uma voz clonada para reaproveitar depois sem reenviar referência."""
    safe_name = (name or "Voz sem nome").strip()[:80]
    voice_id = uuid.uuid4().hex[:12]
    folder = os.path.join(VOICES_DIR, voice_id)
    os.makedirs(folder, exist_ok=True)

    ext = os.path.splitext(ref_audio_path)[1] or ".wav"
    dest = os.path.join(folder, f"ref{ext}")
    shutil.copyfile(ref_audio_path, dest)

    info = {
        "id": voice_id,
        "name": safe_name,
        "language": language,
        "ref_text": ref_text or "",
        "created_at": now_str(),
    }
    with open(os.path.join(folder, "info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    return info

def get_saved_voice(voice_id):
    safe_id = re.sub(r"[^a-zA-Z0-9_-]", "", voice_id or "")
    info_file = voice_info_path(safe_id)
    ref = voice_ref_path(safe_id)
    if not safe_id or not os.path.exists(info_file) or not ref or not os.path.exists(ref):
        return None
    with open(info_file, "r", encoding="utf-8") as f:
        info = json.load(f)
    info["ref_audio_path"] = ref
    return info


def save_wav_file(waveform, sr, file_id):
    path = os.path.join(OUTPUT_DIR, f"{file_id}.wav")
    with wave.open(path, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(waveform.tobytes())
    return path

# Funções utilitárias de áudio
def to_wav_b64(waveform, sr):
    buf = _io.BytesIO()
    with wave.open(buf, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(waveform.tobytes())
    return base64.b64encode(buf.getvalue()).decode()

def generate_speech(text, language, ref_audio_path, instruct,
                     num_step, guidance_scale, denoise,
                     speed, duration, preprocess_prompt, postprocess_output,
                     mode="clone", ref_text=None):
    if not text or not text.strip():
        return None, "Digite algum texto."

    lang_code = None
    if language and language != "Auto":
        lang_code = language.split("(")[-1].rstrip(")").strip() if "(" in language else language

    gen_config = OmniVoiceGenerationConfig(
        num_step=int(num_step or 32),
        guidance_scale=float(guidance_scale) if guidance_scale is not None else 2.0,
        denoise=bool(denoise) if denoise is not None else True,
        preprocess_prompt=bool(preprocess_prompt),
        postprocess_output=bool(postprocess_output),
    )

    kw = dict(text=text.strip(), language=lang_code, generation_config=gen_config)
    if speed is not None and float(speed) != 1.0:
        kw["speed"] = float(speed)
    if duration is not None and float(duration) > 0:
        kw["duration"] = float(duration)

    if mode == "clone":
        if not ref_audio_path:
            return None, "Envie um áudio de referência."
        set_job("generating", "Criando prompt de clonagem da voz de referência...", 35, mode=mode)
        kw["voice_clone_prompt"] = model.create_voice_clone_prompt(ref_audio=ref_audio_path, ref_text=ref_text)
    elif mode == "design" and instruct and instruct.strip():
        kw["instruct"] = instruct.strip()

    try:
        set_job("generating", "OmniVoice gerando áudio. Aguarde...", 55, mode=mode)
        audio = model.generate(**kw)
    except Exception as e:
        return None, f"Erro: {type(e).__name__}: {e}"

    set_job("generating", "Processando waveform gerado...", 78, mode=mode)
    waveform = audio[0].squeeze()
    if hasattr(waveform, "detach"):
        waveform = waveform.detach().cpu().numpy()
    elif hasattr(waveform, "numpy"):
        waveform = waveform.numpy()
    waveform = (waveform * 32767).astype(np.int16)
    dur = waveform.shape[-1] / sampling_rate
    return waveform, f"Áudio de {dur:.1f}s gerado com sucesso"

def gerar_audio_sync(job_id, text, language, ref_path, instruct, num_step,
                     guidance_scale, denoise, speed, duration, mode, ref_text):
    """Roda a geração pesada fora do event loop para /api/status continuar respondendo."""
    set_job("generating", "Preparando parâmetros de geração...", 20, job_id=job_id, mode=mode)

    waveform, status = generate_speech(
        text=text, language=language,
        ref_audio_path=ref_path,
        instruct=instruct or None,
        num_step=num_step, guidance_scale=guidance_scale,
        denoise=denoise, speed=speed, duration=duration,
        preprocess_prompt=True, postprocess_output=True,
        mode=mode, ref_text=ref_text or None,
    )

    if waveform is None:
        return None, status, None, None, None

    set_job("generating", "Convertendo áudio para WAV 44.1kHz...", 88, job_id=job_id, mode=mode)

    output_sr = 44100
    num_samples = int(len(waveform) * output_sr / sampling_rate)
    waveform = resample(waveform, num_samples).astype(np.int16)

    file_path = save_wav_file(waveform, output_sr, job_id)
    audio_b64 = to_wav_b64(waveform, output_sr)
    duration_seconds = len(waveform) / output_sr

    return waveform, status, audio_b64, file_path, duration_seconds

# Servidor FastAPI
app = FastAPI(title="OmniVoice API Premium 44.1kHz Cloudflare")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def root():
    return {
        "status": "online",
        "modelo": "OmniVoice",
        "endpoint": "/api/gerar",
        "health": "/health",
        "status_endpoint": "/api/status",
        "historico_endpoint": "/api/historico",
        "transcrever_endpoint": "/api/transcrever",
        "vozes_endpoint": "/api/vozes",
        "audio_para_voz_endpoint": "/api/audio-para-voz",
        "gemini_endpoint": "/api/melhorar-texto",
        "sample_rate_saida": 44100,
        "is_generating": IS_GENERATING,
        "model_cache_ready": MODEL_CACHE_READY,
    }

@app.get("/health")
async def health():
    return {
        "ok": True,
        "status": "online",
        "is_generating": IS_GENERATING,
        "model_cache_ready": MODEL_CACHE_READY,
        "time": now_str(),
    }

@app.get("/api/status")
async def api_status():
    return {
        "ok": True,
        "job": CURRENT_JOB,
        "is_generating": IS_GENERATING,
        "model_cache_ready": MODEL_CACHE_READY,
        "history_count": len(HISTORY),
        "time": now_str(),
    }

@app.get("/api/historico")
async def api_historico():
    cleanup_outputs()
    public_items = []
    for item in HISTORY:
        public_items.append({
            "id": item.get("id"),
            "mode": item.get("mode"),
            "text_preview": item.get("text_preview"),
            "status": item.get("status"),
            "duration": item.get("duration"),
            "created_at": item.get("created_at"),
            "sample_rate": item.get("sample_rate"),
            "audio_url": f"/api/audio/{item.get('id')}" if item.get("file_exists") else None,
            "voice_name": item.get("voice_name"),
            "transcript": item.get("transcript"),
        })
    return {"ok": True, "items": public_items}

@app.get("/api/audio/{file_id}")
async def api_audio(file_id: str):
    safe_id = re.sub(r"[^a-zA-Z0-9_-]", "", file_id)
    path = os.path.join(OUTPUT_DIR, f"{safe_id}.wav")
    if not os.path.exists(path):
        return JSONResponse({"ok": False, "erro": "Arquivo não encontrado ou já limpo."}, status_code=404)
    return FileResponse(path, media_type="audio/wav", filename=f"audio_{safe_id}.wav")


@app.post("/api/transcrever")
async def api_transcrever(
    audio: UploadFile = File(...),
    language: str = Form("Auto"),
):
    """Transcreve um áudio enviado e retorna texto para o template preencher/revisar."""
    job_id = uuid.uuid4().hex[:12]
    temp_path = None
    try:
        set_job("transcribing", "Recebendo áudio para transcrição...", 5, job_id=job_id, mode="transcribe")
        ext = os.path.splitext(audio.filename or "")[1] or ".wav"
        temp_path = f"/tmp/transcribe_{job_id}{ext}"
        with open(temp_path, "wb") as f:
            f.write(await audio.read())

        text = await asyncio.to_thread(transcribe_audio_file, temp_path, language)
        if not text:
            return JSONResponse({"ok": False, "erro": "Não foi possível transcrever o áudio."}, status_code=400)

        set_job("done", "Transcrição concluída.", 100, job_id=job_id, mode="transcribe")
        return {"ok": True, "text": text, "job_id": job_id}

    except Exception as e:
        set_job("error", f"Erro na transcrição: {repr(e)}", 100, job_id=job_id, mode="transcribe")
        return JSONResponse({"ok": False, "erro": f"Erro na transcrição: {repr(e)}"}, status_code=500)
    finally:
        if temp_path and os.path.exists(temp_path):
            try: os.remove(temp_path)
            except Exception: pass

@app.get("/api/vozes")
async def api_vozes():
    """Lista vozes salvas no runtime atual do Colab."""
    return {"ok": True, "items": list_saved_voices()}

@app.post("/api/vozes/salvar")
async def api_salvar_voz(
    name: str = Form(...),
    language: str = Form("Auto"),
    ref_text: str = Form(""),
    ref_audio: UploadFile = File(...),
):
    """Salva uma voz de referência para usar depois em /api/audio-para-voz ou /api/gerar."""
    job_id = uuid.uuid4().hex[:12]
    temp_path = None
    try:
        set_job("preparing", "Salvando voz clonada...", 10, job_id=job_id, mode="voice_save")
        ext = os.path.splitext(ref_audio.filename or "")[1] or ".wav"
        temp_path = f"/tmp/save_voice_{job_id}{ext}"
        with open(temp_path, "wb") as f:
            f.write(await ref_audio.read())

        # Se o usuário não informar transcrição, tentamos transcrever para melhorar a fidelidade depois.
        final_ref_text = (ref_text or "").strip()
        if not final_ref_text:
            try:
                final_ref_text = await asyncio.to_thread(transcribe_audio_file, temp_path, language)
            except Exception:
                final_ref_text = ""

        info = await asyncio.to_thread(save_voice_profile, name, temp_path, final_ref_text, language)
        set_job("done", f"Voz '{info['name']}' salva com sucesso.", 100, job_id=job_id, mode="voice_save")
        return {"ok": True, "voice": info}

    except Exception as e:
        set_job("error", f"Erro ao salvar voz: {repr(e)}", 100, job_id=job_id, mode="voice_save")
        return JSONResponse({"ok": False, "erro": f"Erro ao salvar voz: {repr(e)}"}, status_code=500)
    finally:
        if temp_path and os.path.exists(temp_path):
            try: os.remove(temp_path)
            except Exception: pass

@app.post("/api/audio-para-voz")
async def api_audio_para_voz(
    input_audio: UploadFile = File(...),
    voice_id: str = Form(...),
    language: str = Form("Auto"),
    num_step: int = Form(32),
    guidance_scale: float = Form(3.0),
    denoise: float = Form(0.8),
    speed: float = Form(1.0),
    duration: float = Form(0),
):
    """
    Converte áudio falado para a voz salva:
    áudio de entrada -> Whisper transcreve -> OmniVoice gera com a voz escolhida.
    """
    global IS_GENERATING

    cleanup_outputs()

    if not GENERATION_LOCK.acquire(blocking=False):
        return JSONResponse({
            "ok": False,
            "erro": "Outra geração está em andamento. Aguarde finalizar e tente novamente."
        }, status_code=429)

    job_id = uuid.uuid4().hex[:12]
    input_path = None

    try:
        IS_GENERATING = True
        set_job("preparing", "Recebendo áudio de entrada...", 5, job_id=job_id, mode="audio_to_voice")

        voice = get_saved_voice(voice_id)
        if not voice:
            return JSONResponse({"ok": False, "erro": "Voz salva não encontrada."}, status_code=404)

        ext = os.path.splitext(input_audio.filename or "")[1] or ".wav"
        input_path = f"/tmp/input_audio_to_voice_{job_id}{ext}"
        with open(input_path, "wb") as f:
            f.write(await input_audio.read())

        set_job("transcribing", "Transcrevendo áudio de entrada...", 18, job_id=job_id, mode="audio_to_voice")
        transcript = await asyncio.to_thread(transcribe_audio_file, input_path, language)

        if not transcript:
            return JSONResponse({"ok": False, "erro": "Não foi possível transcrever o áudio de entrada."}, status_code=400)

        set_job("generating", "Gerando o texto transcrito com a voz selecionada...", 35, job_id=job_id, mode="audio_to_voice")

        result = await asyncio.to_thread(
            gerar_audio_sync,
            job_id, transcript, language, voice["ref_audio_path"], None,
            num_step, guidance_scale, denoise, speed, duration, "clone", voice.get("ref_text") or None
        )

        waveform, status, audio_b64, file_path, duration_seconds = result

        if waveform is None:
            set_job("error", status, 100, job_id=job_id, mode="audio_to_voice")
            return JSONResponse({"ok": False, "erro": status, "transcript": transcript, "job_id": job_id}, status_code=400)

        HISTORY.insert(0, {
            "id": job_id,
            "mode": "audio_to_voice",
            "text_preview": transcript[:120],
            "status": status,
            "duration": round(float(duration_seconds or 0), 1),
            "created_at": now_str(),
            "sample_rate": 44100,
            "file_path": file_path,
            "file_exists": True,
            "voice_id": voice.get("id"),
            "voice_name": voice.get("name"),
            "transcript": transcript,
        })
        cleanup_outputs()

        set_job("done", status, 100, job_id=job_id, mode="audio_to_voice")

        return JSONResponse({
            "ok": True,
            "status": status,
            "audio_b64": audio_b64,
            "sample_rate": 44100,
            "job_id": job_id,
            "download_url": f"/api/audio/{job_id}",
            "transcript": transcript,
            "voice": {"id": voice.get("id"), "name": voice.get("name")},
        })

    except Exception as e:
        set_job("error", f"Erro inesperado: {repr(e)}", 100, job_id=job_id, mode="audio_to_voice")
        return JSONResponse({"ok": False, "erro": f"Erro inesperado: {repr(e)}", "job_id": job_id}, status_code=500)

    finally:
        IS_GENERATING = False
        GENERATION_LOCK.release()
        if input_path and os.path.exists(input_path):
            try: os.remove(input_path)
            except Exception: pass


@app.post("/api/melhorar-texto")
async def melhorar_texto(payload: dict = Body(...)):
    """Melhora texto de narração usando Gemini. Não afeta geração de voz."""
    texto = str(payload.get("text", "")).strip()
    objetivo = str(payload.get("objetivo", "narração")).strip()

    if not texto:
        return JSONResponse({"ok": False, "erro": "Envie um texto para melhorar."}, status_code=400)

    if not GEMINI_API_KEY:
        return JSONResponse({
            "ok": False,
            "erro": "GEMINI_API_KEY não configurada no notebook Colab."
        }, status_code=400)

    prompt = f"""
Você é um roteirista profissional de narração.
Melhore o texto abaixo para narração em português do Brasil.

Regras:
- Preserve a ideia original.
- Deixe o texto mais claro, envolvente e natural para voz.
- Não invente fatos específicos.
- Não use markdown.
- Não coloque títulos.
- Não coloque aspas.
- Retorne apenas o texto final pronto para narração.
- Objetivo: {objetivo}

Texto original:
{texto}
""".strip()

    try:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"
        r = requests.post(
            url,
            headers={
                "Content-Type": "application/json",
                "x-goog-api-key": GEMINI_API_KEY,
            },
            json={
                "contents": [
                    {
                        "role": "user",
                        "parts": [{"text": prompt}]
                    }
                ],
                "generationConfig": {
                    "temperature": 0.7,
                    "topP": 0.9,
                    "maxOutputTokens": 1200
                }
            },
            timeout=60,
        )

        if r.status_code != 200:
            return JSONResponse({
                "ok": False,
                "erro": f"Gemini retornou status {r.status_code}",
                "detalhe": r.text[:500],
            }, status_code=500)

        data = r.json()
        melhorado = (
            data.get("candidates", [{}])[0]
                .get("content", {})
                .get("parts", [{}])[0]
                .get("text", "")
                .strip()
        )

        if not melhorado:
            return JSONResponse({"ok": False, "erro": "Gemini não retornou texto."}, status_code=500)

        return {"ok": True, "text": melhorado}

    except Exception as e:
        return JSONResponse({"ok": False, "erro": f"Erro Gemini: {repr(e)}"}, status_code=500)

@app.post("/api/gerar")
async def gerar(
    text:           str   = Form(...),
    mode:           str   = Form("clone"),
    language:       str   = Form("Auto"),
    ref_text:       str   = Form(""),
    instruct:       str   = Form(""),
    num_step:       int   = Form(32),
    guidance_scale: float = Form(3.0),
    denoise:        float = Form(0.8),
    speed:          float = Form(1.0),
    duration:       float = Form(0),
    ref_audio: UploadFile = File(None),
    voice_id: str = Form(""),
):
    global IS_GENERATING

    cleanup_outputs()

    if not GENERATION_LOCK.acquire(blocking=False):
        return JSONResponse({
            "ok": False,
            "erro": "Outra geração está em andamento. Aguarde finalizar e tente novamente."
        }, status_code=429)

    job_id = uuid.uuid4().hex[:12]
    ref_path = None

    try:
        IS_GENERATING = True
        set_job("preparing", "Recebendo dados e preparando geração...", 8, job_id=job_id, mode=mode)

        saved_voice = None
        if voice_id:
            saved_voice = get_saved_voice(voice_id)
            if not saved_voice:
                return JSONResponse({"ok": False, "erro": "Voz salva não encontrada."}, status_code=404)
            ref_path = saved_voice["ref_audio_path"]
            if not ref_text:
                ref_text = saved_voice.get("ref_text") or ""

        elif ref_audio and ref_audio.filename:
            ext = os.path.splitext(ref_audio.filename)[1] or ".wav"
            ref_path = f"/tmp/ref_{job_id}{ext}"
            with open(ref_path, "wb") as f:
                f.write(await ref_audio.read())

        result = await asyncio.to_thread(
            gerar_audio_sync,
            job_id, text, language, ref_path, instruct,
            num_step, guidance_scale, denoise, speed, duration, mode, ref_text
        )

        waveform, status, audio_b64, file_path, duration_seconds = result

        if waveform is None:
            set_job("error", status, 100, job_id=job_id, mode=mode)
            return JSONResponse({"ok": False, "erro": status, "job_id": job_id}, status_code=400)

        HISTORY.insert(0, {
            "id": job_id,
            "mode": mode,
            "text_preview": text.strip()[:120],
            "status": status,
            "duration": round(float(duration_seconds or 0), 1),
            "created_at": now_str(),
            "sample_rate": 44100,
            "file_path": file_path,
            "file_exists": True,
            "voice_id": saved_voice.get("id") if 'saved_voice' in locals() and saved_voice else None,
            "voice_name": saved_voice.get("name") if 'saved_voice' in locals() and saved_voice else None,
        })
        cleanup_outputs()

        set_job("done", status, 100, job_id=job_id, mode=mode)

        return JSONResponse({
            "ok": True,
            "status": status,
            "audio_b64": audio_b64,
            "sample_rate": 44100,
            "job_id": job_id,
            "download_url": f"/api/audio/{job_id}",
        })

    except Exception as e:
        set_job("error", f"Erro inesperado: {repr(e)}", 100, job_id=job_id, mode=mode)
        return JSONResponse({"ok": False, "erro": f"Erro inesperado: {repr(e)}", "job_id": job_id}, status_code=500)

    finally:
        IS_GENERATING = False
        GENERATION_LOCK.release()
        if ref_path and os.path.exists(ref_path):
            try:
                os.remove(ref_path)
            except Exception:
                pass

PORT = 8000

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

def enviar_url_template(api_endpoint):
    """Envia a URL final /api/gerar para set_api.php e atualiza api_url.json."""
    global LAST_API_ENDPOINT_SENT

    if not SET_API_URL or "SEU-SITE.com" in SET_API_URL:
        print("⚠️ SET_API_URL ainda não configurada. URL não enviada para o template.")
        return False

    if not SET_API_TOKEN or SET_API_TOKEN == "SENHA_FORTE_AQUI":
        print("⚠️ SET_API_TOKEN ainda não configurado. URL não enviada para o template.")
        return False

    if api_endpoint == LAST_API_ENDPOINT_SENT:
        print("ℹ️ URL já enviada anteriormente. Não reenviando.")
        return True

    try:
        r = requests.post(
            SET_API_URL,
            data={"token": SET_API_TOKEN, "url": api_endpoint},
            timeout=25,
        )
        print("📡 Resposta set_api.php:", r.status_code, r.text[:300])

        if r.status_code == 200:
            LAST_API_ENDPOINT_SENT = api_endpoint
            print("✅ URL enviada automaticamente para o template:", api_endpoint)
            return True

        print("⚠️ Falha ao enviar URL para template.")
        return False
    except Exception as e:
        print("⚠️ Erro ao enviar URL para template:", repr(e))
        return False

def drain_stream(stream, out_queue):
    try:
        for line in iter(stream.readline, ''):
            if not line:
                break
            out_queue.put(line)
    except Exception:
        pass

def start_cloudflare_tunnel():
    """Inicia Quick Tunnel e retorna (processo, public_url)."""
    cmd = [
        "cloudflared", "tunnel",
        "--url", f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )

    q = queue.Queue()
    threading.Thread(target=drain_stream, args=(proc.stdout, q), daemon=True).start()
    threading.Thread(target=drain_stream, args=(proc.stderr, q), daemon=True).start()

    public_url = None
    lines = []
    start = time.time()

    while time.time() - start < CLOUDFLARE_CREATE_TIMEOUT:
        if proc.poll() is not None:
            break

        try:
            line = q.get(timeout=1)
            lines.append(line.strip())
            m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
            if m:
                public_url = m.group(0).strip()
                break
        except queue.Empty:
            pass

    if not public_url:
        try:
            proc.terminate()
        except Exception:
            pass
        debug = "\n".join(lines[-20:])
        raise RuntimeError("Não foi possível obter URL trycloudflare.com.\n" + debug)

    return proc, public_url

def stop_process(proc):
    if not proc:
        return
    try:
        proc.terminate()
        time.sleep(3)
        if proc.poll() is None:
            proc.kill()
    except Exception:
        pass

def render_status(public_url, api_endpoint, msg_extra=""):
    clear_output(wait=True)
    display.display(display.HTML(f"""
    <div style='background:#09050f;padding:40px;border-radius:18px;font-family:system-ui,sans-serif;
                max-width:760px;margin:0 auto;border:1px solid #7c3aed;
                box-shadow:0 18px 60px rgba(124,58,237,.18);'>
      <p style='color:#c4b5fd;font-size:11px;letter-spacing:2px;text-transform:uppercase;margin:0 0 16px;font-weight:bold;'>
        🚀 ENDPOINT PÚBLICO PREMIUM VIA CLOUDFLARE TUNNEL</p>

      <div style='background:#12091f;border:1px solid #2e1065;border-radius:10px;padding:14px 16px;margin-bottom:16px;'>
        <p style='color:#22c55e;font-size:10px;letter-spacing:1px;text-transform:uppercase;margin:0 0 6px;font-weight:bold;'>MÉTODO: POST</p>
        <code style='color:#f5f3ff;font-size:13px;word-break:break-all;'>{api_endpoint}</code>
      </div>

      <div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin-bottom:16px;'>
        <div style='background:#052e16;border:1px solid #166534;border-radius:10px;padding:14px;'>
          <p style='color:#86efac;font-size:13px;margin:0 0 6px;font-weight:bold;'>✅ URL enviada ao template</p>
          <p style='color:#86efac99;font-size:12px;margin:0;'>{now_str()}</p>
        </div>
        <div style='background:#1e1b4b;border:1px solid #3730a3;border-radius:10px;padding:14px;'>
          <p style='color:#c7d2fe;font-size:13px;margin:0 0 6px;font-weight:bold;'>🧠 Cache OmniVoice</p>
          <p style='color:#c7d2fe99;font-size:12px;margin:0;'>Modelo carregado 1 vez na GPU</p>
        </div>
        <div style='background:#3b0764;border:1px solid #7e22ce;border-radius:10px;padding:14px;'>
          <p style='color:#e9d5ff;font-size:13px;margin:0 0 6px;font-weight:bold;'>📊 Status e histórico</p>
          <p style='color:#e9d5ff99;font-size:12px;margin:0;'>/api/status e /api/historico ativos</p>
        </div>
      </div>

      <div style='background:#0f172a;border-left:4px solid #8b5cf6;padding:12px;margin-bottom:16px;border-radius:0 8px 8px 0;'>
        <p style='color:#ddd6fe;font-size:12px;margin:0;line-height:1.7;'>
          Cloudflare Tunnel não usa a tela do Localtunnel que pede IP para continuar.
          O monitor não recria o túnel enquanto a voz está sendo gerada.
        </p>
      </div>

      <p style='color:#9ca3af;font-size:12px;margin:0;line-height:1.8;'>
        • URL base: <code>{public_url}</code><br>
        • Health check: <code>{public_url.rstrip("/")}/health</code><br>
        • Status: <code>{public_url.rstrip("/")}/api/status</code><br>
        • Histórico: <code>{public_url.rstrip("/")}/api/historico</code><br>
        • Transcrever: <code>{public_url.rstrip("/")}/api/transcrever</code><br>
        • Vozes: <code>{public_url.rstrip("/")}/api/vozes</code><br>
        • Áudio para voz: <code>{public_url.rstrip("/")}/api/audio-para-voz</code><br>
        • Gemini: <code>{public_url.rstrip("/")}/api/melhorar-texto</code><br>
        • Mantenha esta célula em execução enquanto estiver usando.
      </p>

      {msg_extra}
    </div>
    """))

def create_tunnel_with_retry():
    attempt = 0
    while True:
        attempt += 1
        try:
            proc, public_url = start_cloudflare_tunnel()
            api_endpoint = public_url.rstrip("/") + "/api/gerar"
            enviar_url_template(api_endpoint)
            render_status(public_url, api_endpoint)
            return proc, public_url, api_endpoint
        except Exception as e:
            wait = min(60 * attempt, 300)
            print(f"⚠️ Falha ao criar Cloudflare Tunnel: {repr(e)}")
            print(f"⏳ Tentando novamente em {wait}s...")
            time.sleep(wait)

# Inicia o primeiro túnel.
cf_proc, public_url, api_endpoint = create_tunnel_with_retry()

fail_count = 0

try:
    while True:
        time.sleep(HEALTH_INTERVAL)

        # Durante geração de áudio, não reinicia túnel.
        if IS_GENERATING:
            print(f"🎙️ {now_str()} - OmniVoice gerando áudio. Monitor pausado para evitar troca de URL.")
            continue

        cleanup_outputs()
        health_url = public_url.rstrip("/") + "/health"

        try:
            r = requests.get(health_url, timeout=HEALTH_TIMEOUT)
            if r.status_code == 200:
                fail_count = 0
                print(f"✅ {now_str()} - Health OK:", r.status_code)
                continue

            fail_count += 1
            print(f"⚠️ {now_str()} - Health status inesperado {r.status_code}. Falhas: {fail_count}/{MAX_FAILS}")

        except Exception as e:
            fail_count += 1
            print(f"⚠️ {now_str()} - Health falhou: {repr(e)}. Falhas: {fail_count}/{MAX_FAILS}")

        if fail_count >= MAX_FAILS:
            print("⚠️ Cloudflare Tunnel parece realmente fora. Recriando com backoff...")
            stop_process(cf_proc)
            time.sleep(RECREATE_DELAY)

            cf_proc, public_url, api_endpoint = create_tunnel_with_retry()
            fail_count = 0

except KeyboardInterrupt:
    print("\nServidor finalizado pelo usuário.")
    stop_process(cf_proc)


: 